# 🎸 Ukulele Tablature Generator
**Physics-aware polyphonic ukulele tab generation from raw audio.**

Pipeline:
1. FFT-based brute-force harmonic template matching → top-N note multiset candidates
2. Differentiable Karplus-Strong optimizer → fits physical parameters (decay, inharmonicity) per candidate
3. Combined spectral + physics score → best chord shape wins

> **GPU toggle:** Set `USE_GPU = True` in the *Device Configuration* cell below to run on Kaggle GPU.


## 1 · Install dependencies

In [ ]:
%%capture
!pip install librosa soundfile tqdm

## 2 · Clone / mount the project

Choose **one** of the cells below:
- **Option A (Kaggle Dataset):** Upload the repo as a Kaggle dataset and mount it.
- **Option B (GitHub):** Clone directly from GitHub (requires internet ON in notebook settings).

In [ ]:
# Option B — GitHub clone (enable Internet in Kaggle settings first)
import os
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"  # ← replace
if not os.path.exists("ukulele_tabs"):
    !git clone {REPO_URL} ukulele_tabs
os.chdir("ukulele_tabs/prototype")
print("Working directory:", os.getcwd())

## 3 · Device Configuration ⚙️

**Change `USE_GPU` here — this is the only place you need to touch.**

In [ ]:
# ─────────────────────────────────────────────
USE_GPU = True   # ← True = GPU (Kaggle T4/P100), False = CPU
# ─────────────────────────────────────────────

# Patch device_config at runtime so all modules pick it up
import device_config
import torch

if USE_GPU and torch.cuda.is_available():
    device_config.DEVICE = torch.device("cuda")
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
else:
    device_config.DEVICE = torch.device("cpu")
    if USE_GPU:
        print("⚠️  USE_GPU=True but no CUDA found — using CPU")
    else:
        print("ℹ️  Using CPU")

DEVICE = device_config.DEVICE
print(f"Device: {DEVICE}")

## 4 · Upload your audio file

In [ ]:
from google.colab import files  # remove this line if running on Kaggle
# On Kaggle: add your .wav as a dataset attachment instead

# uploaded = files.upload()  # Colab only
# AUDIO_FILE = list(uploaded.keys())[0]

# Kaggle: point to your input file
AUDIO_FILE = "/kaggle/input/YOUR_DATASET/your_chord.wav"   # ← replace
print(f"Audio: {AUDIO_FILE}")

## 5 · Configure the run

In [ ]:
# ── Pipeline parameters ────────────────────────────────────────────────────
STRUMMED_STRINGS = [1, 2, 3, 4]   # which strings were strummed (1=a, 2=e, 3=c, 4=g)
NUM_NOTES        = 4               # number of simultaneous notes
NUM_ITERATIONS   = 50              # optimizer iterations per candidate (50 = good balance)
LR               = 0.5             # Adam learning rate
TOP_N            = 16              # number of spectral candidates to evaluate
AUDIO_WEIGHT     = 0.5             # candidate output audio comparison weight
SPECTRAL_WEIGHT  = 0.5             # spectral template score weight
# ──────────────────────────────────────────────────────────────────────────

## 6 · Run the pipeline

In [ ]:
# ── Pipeline parameters ────────────────────────────────────────────────────
STRUMMED_STRINGS = [1, 2, 3, 4]   # which strings were strummed (1=a, 2=e, 3=c, 4=g)
NUM_NOTES        = 4               # number of simultaneous notes
NUM_ITERATIONS   = 50              # optimizer iterations per candidate (50 = good balance)
LR               = 0.5             # Adam learning rate
TOP_N            = 16              # number of spectral candidates to evaluate
AUDIO_WEIGHT     = 0.5             # candidate output audio comparison weight
SPECTRAL_WEIGHT  = 0.5             # spectral template score weight
# ──────────────────────────────────────────────────────────────────────────

## 7 · Final Rankings & Tab

In [ ]:
STRING_NAMES = {1: 'a', 2: 'e', 3: 'c', 4: 'g'}

print("\n" + "─"*62)
print("  Candidate Rankings (spectral + physics):\n")
for i, c in enumerate(evaluated, 1):
    notes  = ', '.join(midi_to_name(m) for m in c["multiset_midis"])
    marker = "  ◄── WINNER" if i == 1 else ""
    print(f"  #{i:>2}: [{notes}]")
    print(f"       spectral={c['spectral_score_norm']:.3f}  "
          f"physics={c['physics_score_norm']:.3f}  "
          f"combined={c['combined_score']:.3f}{marker}")

print("\n" + "─"*62)
print("  Final Tablature:\n")
mapping = winner["refined"]["mapping"]
for s_idx in sorted(STRUMMED_STRINGS, reverse=True):
    fret = mapping[s_idx]
    print(f"  {STRING_NAMES[s_idx]}:{fret}  ", end="")
print()

if "inharmonicity_scores" in winner["refined"]:
    print("\n  Inharmonicity diagnostics:")
    for s_idx in sorted(STRUMMED_STRINGS, reverse=True):
        d = winner["refined"]["inharmonicity_scores"][s_idx]
        print(f"  {STRING_NAMES[s_idx]}:{d['fret']}  "
              f"obs={d['observed_stretch']:.4f}  "
              f"exp={d['expected_stretch']:.4f}  "
              f"score={d['score']:.3f}")